# 00 — Setup en Google Colab

1. **Runtime → Cambiar tipo de entorno de ejecución → GPU (T4)**
2. Sube `metricas-proyecto-ml-colab.zip` (generado con `scripts/build_colab_zip.ps1`) o monta Google Drive.
3. Ejecuta las celdas de abajo en orden.
4. Luego abre y ejecuta los notebooks `01` → `07` en secuencia.


## Opción A: subir ZIP desde tu PC

In [ ]:
from google.colab import files
import shutil
import zipfile
from pathlib import Path

ZIP_DEST = Path('/content/metricas-proyecto-ml-colab.zip')
PROJECT_NAME = 'metricas-proyecto-ml'
ROOT = Path('/content') / PROJECT_NAME

# Subir ZIP (si cancelas el diálogo, reutiliza el ZIP ya en /content)
uploaded = files.upload()
if uploaded:
    zip_name = next(iter(uploaded))
    ZIP_DEST.write_bytes(uploaded[zip_name])
    print(f'ZIP guardado en: {ZIP_DEST} ({ZIP_DEST.stat().st_size} bytes)')
elif ZIP_DEST.exists():
    print(f'Reutilizando ZIP existente: {ZIP_DEST}')
else:
    raise FileNotFoundError(
        'No hay ZIP en /content. Sube metricas-proyecto-ml-colab.zip.'
    )

zip_path = ZIP_DEST


def is_project_root(path: Path) -> bool:
    return (
        (path / 'pyproject.toml').is_file()
        and (path / 'requirements.txt').is_file()
        and (path / 'data' / 'processed' / 'train.csv').is_file()
    )


def extract_zip(src: Path, dest: Path) -> None:
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    with zipfile.ZipFile(src) as zf:
        for info in zf.infolist():
            name = info.filename.replace('\\', '/')
            if not name or name.endswith('/'):
                if name:
                    (dest / name.rstrip('/')).mkdir(parents=True, exist_ok=True)
                continue
            target = dest / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as fsrc, open(target, 'wb') as fdst:
                fdst.write(fsrc.read())


def find_root(base: Path) -> Path | None:
    if is_project_root(base):
        return base
    nested = base / PROJECT_NAME
    if is_project_root(nested):
        return nested
    for child in base.iterdir():
        if child.is_dir() and is_project_root(child):
            return child
    return None


extract_dir = Path('/content/_zip_extract')
extract_zip(zip_path, extract_dir)
found = find_root(extract_dir)

if found is None:
    print('Entradas del ZIP:', [str(p) for p in extract_dir.rglob('*')][:20])
    raise FileNotFoundError(
        'No se encontró la raíz del proyecto. Regenera el ZIP en tu PC con: '
        'python scripts/build_colab_zip.py'
    )

if ROOT.exists():
    shutil.rmtree(ROOT)
shutil.move(str(found), str(ROOT))
shutil.rmtree(extract_dir, ignore_errors=True)

print('Proyecto en:', ROOT)
print('Contenido:', sorted(p.name for p in ROOT.iterdir()))

## Opción B: montar Google Drive (si ya subiste el zip ahí)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# zip_path = Path('/content/drive/MyDrive/metricas-proyecto-ml-colab.zip')
# extract_zip(zip_path, Path('/content/_zip_extract'))
# found = find_root(Path('/content/_zip_extract'))
# ... mismo bloque de normalización que Opción A

## Instalar dependencias

Ejecuta la celda de abajo y luego **Entorno de ejecución → Reiniciar sesión** antes de continuar.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/metricas-proyecto-ml')
SRC = PROJECT_ROOT / 'src'

assert PROJECT_ROOT.exists(), f'No existe {PROJECT_ROOT}. Ejecuta primero la celda de extracción.'
assert (PROJECT_ROOT / 'pyproject.toml').exists(), 'Falta pyproject.toml.'
assert (SRC / 'discurso_odio').exists(), f'Falta el paquete en {SRC}'

os.chdir(PROJECT_ROOT)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('Directorio actual:', Path.cwd())
print('src en sys.path:', str(SRC) in sys.path)

In [ ]:
# Instala dependencias (numpy limpio + PyTorch alineado + proyecto)
# Al terminar DEBES reiniciar sesión antes de la verificación.
!python /content/metricas-proyecto-ml/scripts/colab_install_deps.py

## ⚠️ Reiniciar sesión

**Entorno de ejecución → Reiniciar sesión** (obligatorio).

Luego ejecuta **solo** las celdas de abajo (no vuelvas a instalar).

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/metricas-proyecto-ml')
SRC = PROJECT_ROOT / 'src'
assert PROJECT_ROOT.exists(), 'Ejecuta primero la celda de extracción.'

os.chdir(PROJECT_ROOT)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('Directorio:', Path.cwd())
print('src en sys.path:', str(SRC) in sys.path)

## Variables de entorno

In [ ]:
import os
os.environ['RUN_PROFILE'] = 'gpu'
os.environ['MAX_STEPS'] = '0'  # 0 = entrenamiento completo
# os.environ['HF_TOKEN'] = 'hf_...'  # solo para datasets gated en notebook 06
print('RUN_PROFILE =', os.environ['RUN_PROFILE'])

## Verificar que todo está listo

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/metricas-proyecto-ml')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Verificar en subproceso limpio (evita caché del kernel)
check = """
import numpy as np
import torch
from transformers import EarlyStoppingCallback
print('numpy', np.__version__, '| torch', torch.__version__, '| transformers OK')
"""
subprocess.check_call([sys.executable, '-c', check])

import numpy as np
import pandas as pd
import torch
from transformers import EarlyStoppingCallback
from discurso_odio.paths import PROJECT_ROOT as PKG_ROOT, TRAIN_CSV, VAL_CSV, TEST_CSV, CORPUS_CSV, CONFIG_PATH
from discurso_odio.data.loaders import load_split

print(f'numpy={np.__version__} | pandas={pd.__version__} | torch={torch.__version__}')
print('EarlyStoppingCallback OK:', EarlyStoppingCallback)

required = [TRAIN_CSV, VAL_CSV, TEST_CSV, CORPUS_CSV, CONFIG_PATH]
missing = [p for p in required if not p.exists()]
assert not missing, f'Faltan archivos: {missing}'

train = load_split('train')
print('PROJECT_ROOT:', PKG_ROOT)
print('Train:', len(train), 'filas')
print('Listo. Ejecuta notebooks 01 → 07 en orden.')

## Diagnóstico del entorno

In [ ]:
import sys
import torch

from discurso_odio.paths import MODELS_DIR, PROJECT_ROOT

print('Python:', sys.version.split()[0])
print('PROJECT_ROOT:', PROJECT_ROOT)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Modelo mBERT:', (MODELS_DIR / 'mbert-sv' / 'model.safetensors').exists())

## Orden de ejecución

| Notebook | Qué hace | Requiere GPU |
|----------|----------|--------------|
| `01_eda.ipynb` | Análisis exploratorio | No |
| `02_preprocesamiento_features.ipynb` | Features y pesos de clase | No |
| `03_baselines_tfidf.ipynb` | Baselines TF-IDF → `models/baseline_*.joblib` | No |
| `04_finetuning_mbert.ipynb` | Fine-tune mBERT → `models/mbert-sv/` | **Sí** |
| `05_metricas_y_comparacion.ipynb` | Métricas y McNemar | No |
| `06_benchmark_externo.ipynb` | Benchmark zero-shot (internet) | No |
| `07_interpretabilidad.ipynb` | LIME + SHAP | No |

**Atajo:** si ya tienes `models/mbert-sv/model.safetensors`, súbelo a `models/mbert-sv/` y salta el notebook 04.

## Ejecutar todos en secuencia (opcional)

Abre cada notebook manualmente, o corre el runner desde la raíz del proyecto:

In [ ]:
# Ejecutar 01 → 07 (puede tardar horas; el 04 necesita GPU)
# !python scripts/run_notebooks_colab.py

# Si ya tienes el modelo entrenado, salta el 04:
# !python scripts/run_notebooks_colab.py --skip 4